In [2]:
import pandas as pd

In [3]:
data_copy = pd.read_csv('/workspaces/group-project-bas-team/data/data_unencoded.csv')

purchases_by_customer = data_copy.groupby('Survey ResponseID')['Category'].apply(list).tolist()
purchases_by_customer

[['FLASH_MEMORY',
  'HEADPHONES',
  'DISHWARE_BOWL',
  'SHAVING_AGENT',
  'COMPUTER_PROCESSOR',
  'ELECTRONIC_CABLE',
  'AMAZON_TABLET',
  'APPAREL_BELT',
  'PORTABLE_ELECTRONIC_DEVICE_STAND',
  'HEADPHONES',
  'IMMERSION_HEATER',
  'HEADPHONES',
  'FLASH_MEMORY',
  'BODY_LUBRICANT',
  'HEADPHONES',
  'INPUT_MOUSE',
  'HEADPHONES',
  'CELLULAR_PHONE_CASE',
  'BODY_POSITIONER',
  'HANDBAG',
  'UNDERPANTS',
  'UNDERPANTS',
  'TAPE_MEASURE',
  'AMAZON_TABLET_ACCESSORY',
  'COMPUTER_DRIVE_OR_STORAGE',
  'TOOTH_CLEANING_AGENT',
  'TOILET_PAPER',
  'MOUTHWASH',
  'ELECTRONIC_CABLE',
  'SCREWDRIVER',
  'EDIBLE_OIL_VEGETABLE',
  'EDIBLE_OIL_VEGETABLE',
  'SIM_CARD',
  'BED_FRAME',
  'HERB',
  'MEAT',
  'DAIRY_BASED_BUTTER',
  'DAIRY_BASED_CHEESE',
  'JERKY',
  'BODY_LUBRICANT',
  'SEXUAL_STIMULATION_DEVICE',
  'FLASH_MEMORY',
  'BODY_LUBRICANT',
  'FLASH_MEMORY',
  'EDIBLE_OIL_VEGETABLE',
  'EDIBLE_OIL_VEGETABLE',
  'SEXUAL_STIMULATION_DEVICE',
  'SEXUAL_STIMULATION_DEVICE',
  'BODY_LUBRICANT'

In [4]:
from mlxtend.preprocessing import TransactionEncoder
from mlxtend.frequent_patterns import apriori
from mlxtend.frequent_patterns import association_rules

encoder = TransactionEncoder()
encoding = encoder.fit_transform(purchases_by_customer)
transaction_data = pd.DataFrame(encoding, columns=encoder.columns_)

frequent_itemsets = apriori(transaction_data, min_support=0.5, use_colnames=True)

rules = association_rules(frequent_itemsets, metric="confidence", min_threshold=0.5, num_itemsets=len(transaction_data.index))

In [6]:
#Creating a new column for Bayesian confidence
total_transactions = len(purchases_by_customer)

#Defining a smoothing parameter m (weight of the prior)
m = 2 

#Reconstructing the raw counts from support
count_X = rules['antecedent support'] * total_transactions
count_X_and_Y = rules['support'] * total_transactions
prior_Y = rules['consequent support'] # P(Y)

#Bayesian Confidence formula
rules['bayesian_confidence'] = (count_X_and_Y + m * prior_Y) / (count_X + m)

In [7]:
#Calculating P(X|Y) and P(X|~Y)
#P(X|Y) = P(X and Y) / P(Y)
p_X_given_Y = rules['support'] / rules['consequent support']

#P(X|~Y) = P(X and ~Y) / P(~Y)
# P(X and ~Y) = P(X) - P(X and Y)
p_X_and_not_Y = rules['antecedent support'] - rules['support']
p_not_Y = 1 - rules['consequent support']
p_X_given_not_Y = p_X_and_not_Y / p_not_Y

#Kemeny-Oppenheim Measure
rules['kemeny_oppenheim'] = (p_X_given_Y - p_X_given_not_Y) / (p_X_given_Y + p_X_given_not_Y)

#Filtering rules by positive confirmation
bayesian_rules = rules[rules['kemeny_oppenheim'] > 0.5].sort_values(by='kemeny_oppenheim', ascending=False)

In [9]:
%pip install pgmpy

/usr/local/python/3.12.1/lib/python3.12/pty.py:95: DeprecationWarning: This process (pid=10223) is multi-threaded, use of forkpty() may lead to deadlocks in the child.
  pid, fd = os.forkpty()


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 41.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 45.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.0/756.0 kB 31.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 915.6/915.6 MB 13.2 MB/s  0:00:26m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 47.5 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 594.3/594.3 MB 18.0 MB/s  0:00:16m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 41.2 MB/s  0:00:006m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 88.0/88.0 MB 44.1 MB/s  0:00:016m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 954.8/954.8 kB 33.3 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 706.8/706.8 MB 10.4 MB/s  0:00:29m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.1/193.1 MB 25.1 MB/s  0:00:07m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 34.3 MB/s  0:00:0

In [13]:
from pgmpy.estimators import HillClimbSearch, BIC

#Encoding the transactions
te = TransactionEncoder()
te_ary = te.fit(purchases_by_customer).transform(purchases_by_customer)
df_encoded = pd.DataFrame(te_ary, columns=te.columns_)

#Searching for the Bayesian Network structure
frequent_categories = df_encoded.sum().sort_values(ascending=False).head(20).index
df_subset = df_encoded[frequent_categories]

hc = HillClimbSearch(df_subset)
best_model = hc.estimate(scoring_method=BIC(df_subset))

#Viewing the discovered dependencies (aka the edges)
print(best_model.edges())

INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'ABIS_BOOK': 'N', 'SHIRT': 'N', 'ELECTRONIC_CABLE': 'N', 'HEADPHONES': 'N', 'CELLULAR_PHONE_CASE': 'N', 'HEALTH_PERSONAL_CARE': 'N', 'CHARGING_ADAPTER': 'N', 'SHOES': 'N', 'PANTS': 'N', 'NUTRITIONAL_SUPPLEMENT': 'N', 'SKIN_MOISTURIZER': 'N', 'BATTERY': 'N', 'SCREEN_PROTECTOR': 'N', 'SOCKS': 'N', 'PORTABLE_ELECTRONIC_DEVICE_COVER': 'N', 'HOME': 'N', 'MEDICATION': 'N', 'BEAUTY': 'N', 'DRINKING_CUP': 'N', 'TOYS_AND_GAMES': 'N'}
INFO:pgmpy: Datatype (N=numerical, C=Categorical Unordered, O=Categorical Ordered) inferred from data: 
 {'ABIS_BOOK': 'N', 'SHIRT': 'N', 'ELECTRONIC_CABLE': 'N', 'HEADPHONES': 'N', 'CELLULAR_PHONE_CASE': 'N', 'HEALTH_PERSONAL_CARE': 'N', 'CHARGING_ADAPTER': 'N', 'SHOES': 'N', 'PANTS': 'N', 'NUTRITIONAL_SUPPLEMENT': 'N', 'SKIN_MOISTURIZER': 'N', 'BATTERY': 'N', 'SCREEN_PROTECTOR': 'N', 'SOCKS': 'N', 'PORTABLE_ELECTRONIC_DEVICE_COVER': 'N', 'HOME': 'N', 'MEDICAT

[('SHIRT', 'PANTS'), ('SHIRT', 'SOCKS'), ('ELECTRONIC_CABLE', 'CHARGING_ADAPTER'), ('ELECTRONIC_CABLE', 'HEADPHONES'), ('HEADPHONES', 'NUTRITIONAL_SUPPLEMENT'), ('CELLULAR_PHONE_CASE', 'ELECTRONIC_CABLE'), ('CELLULAR_PHONE_CASE', 'PORTABLE_ELECTRONIC_DEVICE_COVER'), ('HEALTH_PERSONAL_CARE', 'CELLULAR_PHONE_CASE'), ('HEALTH_PERSONAL_CARE', 'HOME'), ('HEALTH_PERSONAL_CARE', 'SCREEN_PROTECTOR'), ('CHARGING_ADAPTER', 'HEADPHONES'), ('SHOES', 'SCREEN_PROTECTOR'), ('SHOES', 'MEDICATION'), ('PANTS', 'SOCKS'), ('PANTS', 'SHOES'), ('NUTRITIONAL_SUPPLEMENT', 'SKIN_MOISTURIZER'), ('SKIN_MOISTURIZER', 'BEAUTY'), ('SKIN_MOISTURIZER', 'DRINKING_CUP'), ('SCREEN_PROTECTOR', 'CELLULAR_PHONE_CASE'), ('SCREEN_PROTECTOR', 'PORTABLE_ELECTRONIC_DEVICE_COVER'), ('SCREEN_PROTECTOR', 'CHARGING_ADAPTER'), ('SOCKS', 'MEDICATION'), ('SOCKS', 'DRINKING_CUP'), ('SOCKS', 'SHOES'), ('SOCKS', 'HEALTH_PERSONAL_CARE'), ('SOCKS', 'ELECTRONIC_CABLE'), ('HOME', 'BATTERY'), ('HOME', 'TOYS_AND_GAMES'), ('HOME', 'ABIS_BOOK'),